In [1]:
import pandas as pd

# 訓練（train）データとテスト（test）データを読み込む
train_df = pd.read_csv("data/train.csv")
test_df = pd.read_csv("data/test.csv")

# データの行数（データ数）と列数（特徴の種類）を確認
print(f"訓練データのサイズ: {train_df.shape}")
print(f"テストデータのサイズ: {test_df.shape}")

# 訓練データの最初の5行を表示
train_df.head()

訓練データのサイズ: (594194, 21)
テストデータのサイズ: (254655, 20)


,id,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,Male,0,Yes,Yes,29,Yes,No,DSL,Yes,...,Yes,Yes,No,No,One year,Yes,Mailed check,60.10,1653.85,No
1,1,Male,0,Yes,Yes,58,Yes,No,DSL,Yes,...,No,Yes,Yes,No,Two year,No,Credit card (automatic),69.50,3778.20,No
2,2,Male,0,Yes,No,58,Yes,Yes,Fiber optic,No,...,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,100.40,5841.35,No
3,3,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,69.70,70.70,Yes
4,4,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.45,70.45,Yes


In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# データの読み込み
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")

# 1. 基本情報の確認
print(f"Train Shape: {train.shape}")
print(f"Test Shape: {test.shape}")

# 2. 欠損値とデータ型の確認
display(train.info())

# 3. 目的変数（Churn）のバランス確認（クラス不均衡のチェック）
print("\nTarget Distribution:")
print(train['Churn'].value_counts(normalize=True))

Train Shape: (594194, 21)
Test Shape: (254655, 20)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 594194 entries, 0 to 594193
Data columns (total 21 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   id                594194 non-null  int64  
 1   gender            594194 non-null  object 
 2   SeniorCitizen     594194 non-null  int64  
 3   Partner           594194 non-null  object 
 4   Dependents        594194 non-null  object 
 5   tenure            594194 non-null  int64  
 6   PhoneService      594194 non-null  object 
 7   MultipleLines     594194 non-null  object 
 8   InternetService   594194 non-null  object 
 9   OnlineSecurity    594194 non-null  object 
 10  OnlineBackup      594194 non-null  object 
 11  DeviceProtection  594194 non-null  object 
 12  TechSupport       594194 non-null  object 
 13  StreamingTV       594194 non-null  object 
 14  StreamingMovies   594194 non-null  object 
 15  Contract         

None


Target Distribution:
Churn
No     0.774792
Yes    0.225208
Name: proportion, dtype: float64


In [3]:
import optuna
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder

# --- 1. データの読み込みと基本前処理 ---
X = train_df.drop(['id', 'Churn'], axis=1)
y = train_df['Churn'].map({'Yes': 1, 'No': 0})
X_test = test_df.drop(['id'], axis=1)

# --- 2. Feature Engineering (FE v1 & v2) ---
# 数値の組み合わせ
X['AvgMonthlyCharges'] = X['TotalCharges'] / (X['tenure'] + 1)
X_test['AvgMonthlyCharges'] = X_test['TotalCharges'] / (X_test['tenure'] + 1)
X['IsNewCustomer'] = (X['tenure'] == 0).astype(int)
X_test['IsNewCustomer'] = (X_test['tenure'] == 0).astype(int)

services = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
X['TotalServices'] = (train_df[services] == 'Yes').sum(axis=1)
X_test['TotalServices'] = (test_df[services] == 'Yes').sum(axis=1)

# カテゴリの組み合わせ
X['Contract_Payment'] = train_df['Contract'] + "_" + train_df['PaymentMethod']
X_test['Contract_Payment'] = test_df['Contract'] + "_" + test_df['PaymentMethod']
X['Net_Support'] = train_df['InternetService'] + "_" + train_df['TechSupport']
X_test['Net_Support'] = test_df['InternetService'] + "_" + test_df['TechSupport']

# カテゴリ変数の数値化
cat_features = X.select_dtypes(include=['object']).columns
for col in cat_features:
    le = LabelEncoder()
    full_labels = pd.concat([X[col], X_test[col]]).astype(str)
    le.fit(full_labels)
    X[col] = le.transform(X[col].astype(str))
    X_test[col] = le.transform(X_test[col].astype(str))

# --- 3. Optuna によるハイパーパラメータ最適化 ---
def objective(trial):
    param = {
        'objective': 'binary',
        'metric': 'auc',
        'verbosity': -1,
        'random_state': 42,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1),
        'num_leaves': trial.suggest_int('num_leaves', 20, 150),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.4, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.4, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
    }
    
    X_train_sub, X_valid_sub, y_train_sub, y_valid_sub = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    lgb_train = lgb.Dataset(X_train_sub, label=y_train_sub)
    lgb_valid = lgb.Dataset(X_valid_sub, label=y_valid_sub, reference=lgb_train)

    model = lgb.train(
        param, 
        lgb_train, 
        valid_sets=[lgb_valid], 
        valid_names=['valid'],  # ここで名前を 'valid' に固定します
        num_boost_round=1000, 
        callbacks=[lgb.early_stopping(50)]
    )
    
    # これで 'valid' というキーで確実に取得できます
    return model.best_score['valid']['auc']

print("Starting Optuna optimization...")
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20)

# --- 4. 最適なパラメータで 5-Fold K-Fold を実行 ---
print(f"\nOptimization finished. Best AUC: {study.best_value}")
best_params = {
    'objective': 'binary',
    'metric': 'auc',
    'verbosity': -1,
    'random_state': 42,
    **study.best_params # Optunaの結果を自動反映
}

N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(X_test))

for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y)):
    print(f"\n=== Fold {fold + 1} / {N_FOLDS} ===")
    X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
    y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]
    
    lgb_train = lgb.Dataset(X_train, label=y_train)
    lgb_valid = lgb.Dataset(X_valid, label=y_valid, reference=lgb_train)
    
    model = lgb.train(best_params, lgb_train, valid_sets=[lgb_train, lgb_valid],
                      valid_names=['train', 'valid'], num_boost_round=1000,
                      callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)])
    
    oof_preds[valid_idx] = model.predict(X_valid, num_iteration=model.best_iteration)
    test_preds += model.predict(X_test, num_iteration=model.best_iteration) / N_FOLDS

print(f"\nFinal Overall CV AUC with Best Params: {roc_auc_score(y, oof_preds):.5f}")



import xgboost as xgb

# --- 1. XGBoost 用の Optuna 目的関数 ---
def objective_xgb(trial):
    param = {
        'objective': 'binary:logistic',
        'eval_metric': 'auc',
        'verbosity': 0,
        'random_state': 42,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
    }
    
    X_train_sub, X_valid_sub, y_train_sub, y_valid_sub = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    dtrain = xgb.DMatrix(X_train_sub, label=y_train_sub)
    dvalid = xgb.DMatrix(X_valid_sub, label=y_valid_sub)

    model = xgb.train(param, dtrain, num_boost_round=1000, 
                      evals=[(dvalid, 'eval')], early_stopping_rounds=50, verbose_eval=False)
    
    return model.best_score

print("Starting XGBoost Optuna optimization...")
study_xgb = optuna.create_study(direction='maximize')
study_xgb.optimize(objective_xgb, n_trials=15) # 回数は状況に合わせて調整

# --- 2. XGBoost で 5-Fold K-Fold を実行 ---
best_params_xgb = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'random_state': 42,
    **study_xgb.best_params
}

oof_preds_xgb = np.zeros(len(X))
test_preds_xgb = np.zeros(len(X_test))

for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y)):
    print(f"\n=== XGBoost Fold {fold + 1} ===")
    X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
    y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]
    
    dtrain = xgb.DMatrix(X_train, label=y_train)
    dvalid = xgb.DMatrix(X_valid, label=y_valid)
    
    model = xgb.train(best_params_xgb, dtrain, num_boost_round=1000,
                      evals=[(dvalid, 'eval')], early_stopping_rounds=50, verbose_eval=100)
    
    oof_preds_xgb[valid_idx] = model.predict(dvalid)
    test_preds_xgb += model.predict(xgb.DMatrix(X_test)) / N_FOLDS

# --- 3. ブレンディング (LightGBM + XGBoost) ---
# LightGBMの予測（前回の test_preds）と XGBoostの予測を混ぜる
# 比率はCVスコアが良い方を多めにするのが基本です
final_preds = (test_preds * 0.7) + (test_preds_xgb * 0.3)

print(f"\nXGBoost CV AUC: {roc_auc_score(y, oof_preds_xgb):.5f}")
print(f"Blended CV AUC: {roc_auc_score(y, (oof_preds * 0.7) + (oof_preds_xgb * 0.3)):.5f}")






from catboost import CatBoostClassifier, Pool

# --- 1. CatBoost 用の Optuna 目的関数 ---
def objective_cat(trial):
    param = {
        'objective': 'Logloss',
        'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.01, 0.1),
        'depth': trial.suggest_int('depth', 1, 12),
        'boosting_type': trial.suggest_categorical('boosting_type', ['Ordered', 'Plain']),
        'bootstrap_type': trial.suggest_categorical('bootstrap_type', ['Bayesian', 'Bernoulli', 'MVS']),
        'used_ram_limit': '3gb',
        'eval_metric': 'AUC',
        'random_seed': 42,
        'logging_level': 'Silent'
    }

    if param['bootstrap_type'] == 'Bayesian':
        param['bagging_temperature'] = trial.suggest_float('bagging_temperature', 0, 10)
    elif param['bootstrap_type'] == 'Bernoulli':
        param['subsample'] = trial.suggest_float('subsample', 0.1, 1)

    X_train_sub, X_valid_sub, y_train_sub, y_valid_sub = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    model = CatBoostClassifier(**param)
    model.fit(X_train_sub, y_train_sub, eval_set=[(X_valid_sub, y_valid_sub)], early_stopping_rounds=50)
    
    return model.get_best_score()['validation']['AUC']

print("Starting CatBoost Optuna optimization...")
study_cat = optuna.create_study(direction='maximize')
study_cat.optimize(objective_cat, n_trials=10) # まずは10回程度

# --- 2. CatBoost で 5-Fold K-Fold を実行 ---
best_params_cat = {
    'objective': 'Logloss',
    'eval_metric': 'AUC',
    'random_seed': 42,
    'logging_level': 'Info',
    **study_cat.best_params
}

oof_preds_cat = np.zeros(len(X))
test_preds_cat = np.zeros(len(X_test))

for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y)):
    print(f"\n=== CatBoost Fold {fold + 1} ===")
    X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
    y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]
    
    model = CatBoostClassifier(**best_params_cat)
    model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], early_stopping_rounds=50, verbose=100)
    
    oof_preds_cat[valid_idx] = model.predict_proba(X_valid)[:, 1]
    test_preds_cat += model.predict_proba(X_test)[:, 1] / N_FOLDS

# --- 3. 最終ブレンディング (LGBM + XGB + Cat) ---
# CVスコアに基づき重みを調整します（例：LGBM 0.4 / XGB 0.3 / Cat 0.3）
# 重みの微調整（LGBMとXGBを重視）
# w1: LGBM, w2: XGB, w3: CatBoost
w1, w2, w3 = 0.45, 0.45, 0.10 

oof_final_v2 = (oof_preds * w1) + (oof_preds_xgb * w2) + (oof_preds_cat * w3)
final_preds_v2 = (test_preds * w1) + (test_preds_xgb * w2) + (test_preds_cat * w3)

current_cv = roc_auc_score(y, oof_final_v2)
print(f"Adjusted 3-Model CV AUC: {current_cv:.5f}")

if current_cv > 0.91669:
    print("✨ スコア更新！これを提出しましょう。")
else:
    print("🤔 まだ 2-Model (0.91669) の方が高いです。w3 をさらに下げるか、2-Model で提出しましょう。")

[I 2026-03-04 21:22:08,830] A new study created in memory with name: no-name-da40651b-f275-4d7f-990a-d97039d90a99


Starting Optuna optimization...
Training until validation scores don't improve for 50 rounds


[I 2026-03-04 21:22:17,454] Trial 0 finished with value: 0.9165982347423652 and parameters: {'learning_rate': 0.05056741992546216, 'num_leaves': 43, 'feature_fraction': 0.6667193950487235, 'bagging_fraction': 0.7851382599942556, 'bagging_freq': 5, 'min_child_samples': 16}. Best is trial 0 with value: 0.9165982347423652.


Early stopping, best iteration is:
[540]	valid's auc: 0.916598
Training until validation scores don't improve for 50 rounds


[I 2026-03-04 21:22:22,134] Trial 1 finished with value: 0.9159267718480504 and parameters: {'learning_rate': 0.07856982618691115, 'num_leaves': 145, 'feature_fraction': 0.5777681058137711, 'bagging_fraction': 0.8275669000221203, 'bagging_freq': 7, 'min_child_samples': 6}. Best is trial 0 with value: 0.9165982347423652.


...


=== XGBoost Fold 1 ===
[0]	eval-auc:0.89652
[100]	eval-auc:0.91321
[200]	eval-auc:0.91484
[300]	eval-auc:0.91541
[400]	eval-auc:0.91580
[500]	eval-auc:0.91599
[600]	eval-auc:0.91606
[700]	eval-auc:0.91615
[766]	eval-auc:0.91615

=== XGBoost Fold 2 ===
[0]	eval-auc:0.89761
[100]	eval-auc:0.91433
[200]	eval-auc:0.91585
[300]	eval-auc:0.91649
[400]	eval-auc:0.91677
[500]	eval-auc:0.91693
[600]	eval-auc:0.91699
[700]	eval-auc:0.91709
[800]	eval-auc:0.91710
[806]	eval-auc:0.91711

=== XGBoost Fold 3 ===
[0]	eval-auc:0.89806
[100]	eval-auc:0.91379
[200]	eval-auc:0.91540
[300]	eval-auc:0.91608
[400]	eval-auc:0.91640
[500]	eval-auc:0.91660
[600]	eval-auc:0.91670
[700]	eval-auc:0.91677
[800]	eval-auc:0.91682
[900]	eval-auc:0.91688
[999]	eval-auc:0.91689

=== XGBoost Fold 4 ===
[0]	eval-auc:0.89852
[100]	eval-auc:0.91478
[200]	eval-auc:0.91638
[300]	eval-auc:0.91700
[400]	eval-auc:0.91736
[500]	eval-auc:0.91758
[600]	eval-auc:0.91769
[700]	eval-auc:0.91778
[800]	eval-auc:0.91780
[878]	eval-auc:

[I 2026-03-04 21:33:59,818] A new study created in memory with name: no-name-6d422b2a-7e01-4007-8129-b3b7927d6992



XGBoost CV AUC: 0.91659
Blended CV AUC: 0.91676
Starting CatBoost Optuna optimization...


[I 2026-03-04 21:34:42,725] Trial 0 finished with value: 0.9148974955579512 and parameters: {'colsample_bylevel': 0.045881665111874914, 'depth': 6, 'boosting_type': 'Ordered', 'bootstrap_type': 'MVS'}. Best is trial 0 with value: 0.9148974955579512.
[I 2026-03-04 21:35:28,759] Trial 1 finished with value: 0.9151105055053003 and parameters: {'colsample_bylevel': 0.059754834686397966, 'depth': 4, 'boosting_type': 'Ordered', 'bootstrap_type': 'Bernoulli', 'subsample': 0.794952456493547}. Best is trial 1 with value: 0.9151105055053003.
[I 2026-03-04 21:35:40,057] Trial 2 finished with value: 0.9126954011810362 and parameters: {'colsample_bylevel': 0.07711115805642542, 'depth': 1, 'boosting_type': 'Plain', 'bootstrap_type': 'Bayesian', 'bagging_temperature': 3.048601277010273}. Best is trial 1 with value: 0.9151105055053003.
[I 2026-03-04 21:35:51,297] Trial 3 finished with value: 0.9152924944033267 and parameters: {'colsample_bylevel': 0.058634991996651904, 'depth': 7, 'boosting_type': 'Pl


=== CatBoost Fold 1 ===
Learning rate set to 0.145255
Metric AUC is not calculated on train by default. To calculate this metric on train, add hints=skip_train~false to metric parameters.

Contract_Payment, bin=2 score 237.3614693
Contract_Payment, bin=1 score 245.4426157
PhoneService, bin=0 score 246.4858122
Contract_Payment, bin=6 score 247.378053
OnlineBackup, bin=0 score 249.4259981
InternetService, bin=0 score 252.9458426
OnlineBackup, bin=1 score 255.6227244
0:	test: 0.8768065	best: 0.8768065 (0)	total: 18.4ms	remaining: 18.3s

PaymentMethod, bin=1 score 149.2420092
MonthlyCharges, bin=102 score 165.4889498
PaymentMethod, bin=2 score 167.4499671
DeviceProtection, bin=0 score 170.7411446
PhoneService, bin=0 score 170.9268724
TotalServices, bin=3 score 175.2465956
OnlineBackup, bin=1 score 175.5837527


PhoneService, bin=0 score 80.71582282
TotalCharges, bin=109 score 95.61316681
Partner, bin=0 score 99.55550216
Contract, bin=0 score 115.4311513
StreamingMovies, bin=1 score 125.09

In [4]:
# 1. 最終予測値（final_preds_v2）を使ってCSVを作成
submission_final = pd.read_csv("data/sample_submission.csv")
submission_final['Churn'] = final_preds_v2
submission_final.to_csv("submission_3model_final.csv", index=False)

print("Created: submission_3model_final.csv")

Created: submission_3model_final.csv
